# 02 — Preprocesamiento y Cálculo del TRIMP Objetivo

**TFM RunnAing** | TRIMP de Banister (1991)

Objetivos:
- Aplicar el funnel de limpieza definido en el EDA
- Calcular el TRIMP de Banister diferenciado por sexo (b=1.92/1.67)
- La FC SOLO se usa aquí para construir el target; NUNCA entrará en X
- Guardar el dataset limpio con columna `trimp` para los notebooks siguientes

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
from pathlib import Path

from src.data_loader import auto_detect_and_load, filter_running_sessions
from src.trimp import banister_trimp, DEFAULT_HR_REST, DEFAULT_HR_MAX

np.random.seed(42)
DATA_DIR = Path('../data')
PROCESSED = DATA_DIR / 'processed'
PROCESSED.mkdir(parents=True, exist_ok=True)

print('Setup listo')

In [ ]:
# Carga del dataset
df_raw = auto_detect_and_load('../data/raw')
df = filter_running_sessions(df_raw)
print(f'Sesiones de running: {len(df):,}')

In [ ]:
# Filtrar sesiones válidas (gender + heart_rate + GPS)
df = df[df['gender'].notna()]
df = df[df['heart_rate'].apply(lambda x: isinstance(x, list) and len(x) >= 2)]
df = df[df['latitude'].apply(lambda x: isinstance(x, list) and len(x) >= 2)]
df = df[df['altitude'].apply(lambda x: isinstance(x, list) and len(x) >= 2)]
print(f'Sesiones tras filtros: {len(df):,}')

In [ ]:
# Calcular FC media por sesión (SOLO para construir TRIMP target)
df['hr_mean'] = df['heart_rate'].apply(np.nanmean)

# Duración en minutos desde los timestamps
def extract_duration_min(ts_list):
    try:
        ts = np.array(ts_list, dtype=float)
        if ts[-1] - ts[0] > 1e8:  # milliseconds
            ts = ts / 1000.0
        return (ts[-1] - ts[0]) / 60.0
    except Exception:
        return np.nan

df['duration_min'] = df['timestamp'].apply(extract_duration_min)

# Fecha de la sesión (para ACWR más adelante)
def extract_date(ts_list):
    try:
        ts = min(ts_list)
        if ts > 1e9:
            ts /= 1000.0
        return pd.to_datetime(ts, unit='s').date()
    except Exception:
        return None

df['date'] = df['timestamp'].apply(extract_date)

# Eliminar sesiones con duración inválida
df = df[df['duration_min'] > 0].copy()
df = df[df['hr_mean'] > 0].copy()
print(f'Sesiones con duración y FC válidas: {len(df):,}')

In [ ]:
# Calcular TRIMP de Banister (target y)
# NOTA: hr_mean se usa AQUÍ para construir el target, nunca se incluirá en X
df['trimp'] = df.apply(
    lambda row: banister_trimp(
        duration_min=row['duration_min'],
        hr_mean=row['hr_mean'],
        gender=row['gender'],
        hr_rest=DEFAULT_HR_REST,
        hr_max=DEFAULT_HR_MAX,
    ),
    axis=1
)

# Eliminar NaN en TRIMP
df = df[df['trimp'].notna()].copy()
print(f'Sesiones con TRIMP válido: {len(df):,}')

print('\nEstadísticas del TRIMP objetivo:')
print(df['trimp'].describe())

In [ ]:
# Guardar dataset preprocesado
# IMPORTANTE: se guardan las columnas de lista (para feature engineering)
# pero hr_mean se conserva solo como referencia para el target, NO para X
cols_to_save = ['userId', 'gender', 'date', 'duration_min',
                'timestamp', 'latitude', 'longitude', 'altitude',
                'trimp']
# Excluir heart_rate y hr_mean del dataset procesado
df_processed = df[cols_to_save].copy()
df_processed.to_pickle(PROCESSED / 'df_processed.pkl')
print(f'Dataset procesado guardado: {PROCESSED}/df_processed.pkl')
print(f'Columnas: {list(df_processed.columns)}')
assert 'heart_rate' not in df_processed.columns, 'ERROR: FC en dataset procesado'
assert 'hr_mean' not in df_processed.columns, 'ERROR: hr_mean en dataset procesado'
print('OK — FC eliminada del dataset procesado.')